# 07 · Capstone: Finding the Polar Vortex in the Data

**Goal: work like a data detective. Use temperature data to find a real,
documented cold event, then corroborate your finding against the news record.**

The earlier capstone studied a region at a moment in time. This one is an
investigation across time. You will take a documented event, the polar vortex
cold snaps that periodically freeze Chicago, and ask whether the Sage temperature
record shows their fingerprint, on the same dates the news reported.

### What a polar vortex is, stated carefully

The polar vortex is a band of cold air and low pressure that circles each pole
high in the stratosphere. It is always there. What people loosely call "a polar
vortex" is really an Arctic air outbreak: a lobe of that vortex weakens, dislodges,
and slides south, dumping bitterly cold air over the mid-latitudes for several
days. Meteorologists are careful with the term, and so should we be. In the data
we will find an extreme, sustained cold spell. Calling it "the polar vortex" is an
interpretation we borrow from the news and the National Weather Service, not
something temperature data alone can prove.

### The signature we are hunting

An Arctic outbreak leaves a clear mark: a sharp drop to extreme lows that lasts
for several consecutive days, far below the normal winter range, then recovers.
That shape, sustained and deep, is what separates a polar-vortex event from
ordinary day-to-day cold.

One warning up front, which we develop in section 3: the node's own temperature
sensor runs warm, so on live data it may not even register the outdoor cold.
Knowing what your instrument actually measures is the first job of a data
detective.

This notebook runs offline. When there is no network it uses synthetic winter data
modeled as clean outdoor air, with a real cold plunge embedded, so the detection
method is fully demonstrable. With a network you can point it at real January data,
keeping the sensor caveat in mind.

## 1. The news record: documented Chicago cold events

Good detective work starts from a lead, not from staring at noise. Here is the
independent record we will corroborate against, drawn from the National Weather
Service and news reporting. These are the facts; our data job is to see whether
the temperature record agrees.

- **January 2019 polar vortex.** O'Hare fell to -23 F on January 30, 2019, its
  coldest in 34 years, and -21 F on January 31. Wind chills reached about -52 F,
  and the city stayed below zero for roughly 52 straight hours. (National Weather
  Service Chicago; CBS Chicago)
- **January 19 to 24, 2025 cold spell.** A lobe of the polar vortex brought several
  sub-zero days, coldest on the morning of January 21, with air temperatures
  around 5 to 15 below zero and wind chills to -35 F. (National Weather Service
  Chicago)
- **January 2014.** Chicago hit -16 F, popularizing the term "polar vortex" in
  everyday use. (City Cast Chicago)
- **January 2026.** In the most recent outbreak, Chicago logged about -11 F on
  January 23, 2026, with a wind chill near -36 F. (Wikipedia, 2026 North American
  cold wave)

The worked example below uses the January 2025 event, recent enough to be in
current Sage data, but the method is identical for any of these dates.

In [ ]:
# The documented events, as a structure we can compare our detections against.
documentedEvents = [
    {"name": "January 2019 polar vortex", "start": "2019-01-29", "end": "2019-01-31",
     "reportedLowF": -23, "note": "O'Hare -23 F on Jan 30, coldest in 34 years"},
    {"name": "January 2025 cold spell", "start": "2025-01-19", "end": "2025-01-24",
     "reportedLowF": -15, "note": "coldest morning Jan 21, wind chills to -35 F"},
    {"name": "January 2026 outbreak", "start": "2026-01-21", "end": "2026-01-24",
     "reportedLowF": -11, "note": "about -11 F on Jan 23, wind chill near -36 F"},
]

print("documented Chicago cold events we can look for:")
for event in documentedEvents:
    print(f"  {event['start']} to {event['end']}: {event['name']}")
    print(f"      reported low {event['reportedLowF']} F. {event['note']}")

## 2. How to navigate and explore data you do not know

Before any detection code, here is the general method. It applies far beyond this
notebook, to any unfamiliar dataset.

1. **Establish coverage.** Does this node even have data for the period you care
   about? Use the cheap `head` and `tail` probes from notebook `01` to find the
   first and last records before you pull a big window.
2. **Get the big picture.** Pull a wide window, look at summary statistics and a
   quick plot. Learn the normal range and the seasonality before you hunt for
   anything unusual.
3. **Bring outside knowledge.** You rarely find events by scanning noise. Start
   from a hypothesis grounded in something external, here a documented date from
   the news. The lead tells you where to look.
4. **Zoom to the candidate window.** Pull just that period, at full resolution.
5. **Reduce to a signature.** Raw readings are noisy. A polar-vortex event is a
   sustained multi-day drop, so daily minimums over several days capture it far
   better than any single reading.
6. **Quantify and corroborate.** Define the event with an explicit rule, then
   check the dates you found against the independent record. Agreement between two
   sources you trust is what turns a guess into a finding.

The rest of the notebook walks steps 4 through 6 in code.

In [ ]:
import sage_data_client
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams["figure.figsize"] = (12, 6)


def makeWinterBeehive(vsn="W06C", startDate="2025-01-15", days=14, stepMinutes=15,
                      seed=19, vortexCenterDay=6.0, vortexWidthDays=2.0,
                      vortexLowC=-21.0):
    """
    Build a Beehive-schema winter temperature record (Celsius) with a normal cold
    baseline and an embedded polar-vortex plunge, so the detection has a real
    event to find. The plunge is a smooth multi-day dip reaching vortexLowC at its
    coldest, centered vortexCenterDay days after startDate.

    Columns: timestamp, name, value, meta.vsn, meta.sensor. Values are Celsius.
    """
    rng = np.random.default_rng(seed)
    periods = int(days * 24 * 60 / stepMinutes)
    times = pd.date_range(startDate, periods=periods, freq=f"{stepMinutes}min", tz="UTC")

    hourOfDay = times.hour + times.minute / 60.0
    diurnal = 4.0 * np.sin((hourOfDay - 9.0) / 24.0 * 2 * np.pi)   # small winter swing
    baseline = -3.0                                                # typical January cold
    dayIndex = (times - times[0]).days
    wander = np.interp(dayIndex, np.arange(days), rng.normal(0, 2.0, days))
    celsius = baseline + diurnal + wander + rng.normal(0, 0.6, periods)

    # The polar-vortex plunge: a bell-shaped dip over the vortex window.
    dayFloat = dayIndex + hourOfDay / 24.0
    plungeDepth = vortexLowC - baseline
    dip = plungeDepth * np.exp(-((dayFloat - vortexCenterDay) ** 2)
                               / (2 * vortexWidthDays ** 2))
    celsius = celsius + dip

    return pd.DataFrame({
        "timestamp": times,
        "name": "env.temperature",
        "value": celsius,
        "meta.vsn": vsn,
        "meta.sensor": "bme680",
    })


print("winter data generator ready")

## 3. Get the data for the candidate window

We investigate the January 2025 event, so we ask for late January 2025. The
function tries the live Beehive first and falls back to the synthetic winter
record, which embeds the plunge, so this always runs. When live data returns more
than one sensor, we keep the single most common one, as in the earlier capstone.

In [ ]:
def getWinterData(vsn, start, end):
    """Try a live query for the window; fall back to synthetic winter data.
    Returns (frame, isLive)."""
    try:
        liveDf = sage_data_client.query(
            start=start, end=end, filter={"name": "env.temperature", "vsn": vsn})
        if not liveDf.empty:
            if "meta.sensor" in liveDf.columns and liveDf["meta.sensor"].nunique() > 1:
                topSensor = liveDf["meta.sensor"].value_counts().index[0]
                liveDf = liveDf[liveDf["meta.sensor"] == topSensor]
            liveDf["timestamp"] = pd.to_datetime(liveDf["timestamp"])
            print(f"using live Beehive data: {len(liveDf)} readings for {vsn}")
            return liveDf.sort_values("timestamp").reset_index(drop=True), True
    except Exception as err:
        print("live query unavailable:", err)

    print("using synthetic winter data with an embedded polar-vortex plunge")
    return makeWinterBeehive(vsn=vsn), False


winterDf, isLive = getWinterData("W06C", start="2025-01-15T00:00:00Z",
                                 end="2025-01-29T00:00:00Z")
winterDf["valueF"] = winterDf["value"] * 9 / 5 + 32
print(f"window: {winterDf['timestamp'].min()} to {winterDf['timestamp'].max()}")
print(f"overall range: {winterDf['valueF'].min():.1f} to {winterDf['valueF'].max():.1f} F")
winterDf.head()

### A detective's first question: is this thermometer measuring the air?

Before hunting a cold event, confirm your sensor can see it. This matters here
more than usual, and it is the most important lesson in this notebook.

Two things about the numbers you just pulled.

- **Normal January days sit in the 20s and 30s F.** A polar vortex is not a cold
  month, it is a sharp plunge of a few days far below the normal winter range. So
  readings around 30 to 40 F on the days before and after the event are expected.
  The vortex is the dip below zero, not the whole window.
- **The Sage `env.temperature` sensor runs warm.** As notebook 06 showed, this
  sensor sits close to the node electronics and reads well above shaded outdoor
  air, often 80 to 140 F in summer. In winter that same warm bias means a live node
  can report 40 F and up even while the outdoor air is below zero. On real data this
  metric is a poor outdoor thermometer: it understates the cold and can hide the
  vortex entirely.

That is why this notebook's offline data is modeled as clean ambient air, what a
properly sited weather thermometer would read, so you can learn the method on a
signal that actually contains the event. On live nodes, treat `env.temperature` as
enclosure-influenced, and expect the vortex to be muted or invisible unless the
node has a dedicated, externally sited air sensor. The check below makes this
concrete.

In [ ]:
def looksLikeAmbientAir(frame, winterCeilingF=45.0):
    """
    Heuristic check that a winter series looks like outdoor air, not enclosure
    temperature. Real Chicago January air spends plenty of time below ~45 F, so if
    the minimum never drops that low, the series is probably enclosure-influenced.

    Returns (isAmbient, minF, fractionBelowCeiling).
    """
    minF = float(frame["valueF"].min())
    fractionBelow = float((frame["valueF"] < winterCeilingF).mean())
    isAmbient = (minF < winterCeilingF) and (fractionBelow > 0.5)
    return isAmbient, minF, fractionBelow


isAmbient, coldestF, fractionBelow = looksLikeAmbientAir(winterDf)
print(f"coldest reading: {coldestF:.1f} F")
print(f"fraction of readings below 45 F: {fractionBelow:.0%}")
print()
if isAmbient:
    print("This looks like outdoor air: it gets genuinely cold, so a polar vortex")
    print("would be visible. Continue the investigation.")
else:
    print("Warning: this never gets truly cold. It is almost certainly enclosure")
    print("temperature (see notebook 06), not outdoor air, so the polar vortex will")
    print("not show up here. You would need a properly sited external air sensor.")
print(f"\nrunning on {'live' if isLive else 'synthetic ambient'} data")

## 4. Explore: reduce the noise to daily extremes

A reading every fifteen minutes is too noisy to reason about days. Collapse the
record to one row per day holding the minimum, mean, and maximum. The daily
minimum is the key series: an Arctic outbreak is defined by how cold it gets and
how long it stays there, which the daily minimum captures directly.

In [ ]:
def dailyExtremes(frame):
    """Return a DataFrame indexed by date with min, mean, and max temperature (C)."""
    indexed = frame.copy()
    indexed["timestamp"] = pd.to_datetime(indexed["timestamp"])
    indexed = indexed.set_index("timestamp")
    daily = indexed["value"].resample("1D").agg(["min", "mean", "max"]).dropna()
    return daily


daily = dailyExtremes(winterDf)
dailyF = daily * 9 / 5 + 32     # a Fahrenheit view, since the news uses F
print("daily temperature summary (F):")
print(dailyF.round(1).to_string())

## 5. Detect the signature: sustained extreme cold

Now the rule. A single cold reading is weather noise; a polar-vortex event is
several consecutive days whose minimum stays below a severe threshold. We use
-18 C, which is about 0 F, a common marker of a serious Arctic outbreak, and
require at least two days in a row. The function returns every qualifying spell
with its dates, length, and coldest reading.

The threshold is a choice you set from domain knowledge, exactly like the
plausibility band in the earlier capstone. Raise or lower it to define "extreme"
for your climate.

In [ ]:
def detectColdSpells(dailyMin, thresholdC=-18.0, minDays=2):
    """
    Find runs of consecutive days whose daily minimum is at or below thresholdC.

    Args:
        dailyMin: a date-indexed Series of daily minimum temperatures in Celsius.
        thresholdC: the severe-cold cutoff. -18 C is about 0 F.
        minDays: the minimum length of a run to report.

    Returns:
        A list of dicts with startDate, endDate, days, coldestC, coldestF.
    """
    below = dailyMin <= thresholdC

    runs = []
    runStart = None
    prevDate = None
    for date, isCold in below.items():
        if isCold and runStart is None:
            runStart = date
        elif not isCold and runStart is not None:
            runs.append((runStart, prevDate))
            runStart = None
        prevDate = date
    if runStart is not None:
        runs.append((runStart, prevDate))

    spells = []
    for start, end in runs:
        mask = (dailyMin.index >= start) & (dailyMin.index <= end)
        days = int(mask.sum())
        if days >= minDays:
            coldestC = float(dailyMin[mask].min())
            spells.append({
                "startDate": start.date(),
                "endDate": end.date(),
                "days": days,
                "coldestC": round(coldestC, 1),
                "coldestF": round(coldestC * 9 / 5 + 32, 1),
            })
    return spells


coldSpells = detectColdSpells(daily["min"], thresholdC=-18.0, minDays=2)

if coldSpells:
    print("detected sustained cold spells (daily min at or below -18 C / ~0 F):")
    for spell in coldSpells:
        print(f"  {spell['startDate']} to {spell['endDate']}  "
              f"({spell['days']} days), coldest {spell['coldestF']} F")
else:
    print("no sustained sub-threshold cold spell in this window.")

## 6. The payoff: see the event you found

Plot the daily minimum, mean, and maximum across the window, mark the severe-cold
threshold, and shade the spells the detector flagged. The shaded band is the
polar-vortex event, located by the rule rather than by eye.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# The full-resolution trace, faint, so the plunge is visible in detail.
ax.plot(winterDf["timestamp"], winterDf["valueF"], color="tab:blue",
        linewidth=0.4, alpha=0.3)

# Daily min, mean, max.
ax.plot(dailyF.index, dailyF["min"], "o-", color="tab:blue", label="daily min")
ax.plot(dailyF.index, dailyF["mean"], "o-", color="gray", label="daily mean")
ax.plot(dailyF.index, dailyF["max"], "o-", color="tab:orange", label="daily max")

# The severe-cold threshold, -18 C is about 0 F.
ax.axhline(0, color="purple", linestyle="--", linewidth=1, label="~0 F cold threshold")

# Shade each detected spell.
for spell in coldSpells:
    ax.axvspan(pd.Timestamp(spell["startDate"]),
               pd.Timestamp(spell["endDate"]) + pd.Timedelta(days=1),
               color="red", alpha=0.12)

ax.set_xlabel("Date")
ax.set_ylabel("Temperature (F)")
ax.set_title("Chicago temperature, late January 2025, detected cold spell shaded")
ax.legend(loc="upper right")
ax.grid(True, linestyle="--", alpha=0.5)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
for tickLabel in ax.get_xticklabels():
    tickLabel.set_rotation(45)
plt.tight_layout()
plt.show()

## 7. Connect the data to the news

The last step is corroboration. We take each spell the data flagged and check
whether its dates overlap a documented event. Overlap between an independent news
record and our own detection is what turns "the data got cold" into "the data
shows the polar vortex the news reported."

In [ ]:
def overlaps(startA, endA, startB, endB):
    """True if two inclusive date ranges overlap."""
    return startA <= endB and startB <= endA


def matchToNews(spell, events):
    """Return the documented event whose dates overlap the spell, or None."""
    spellStart = pd.Timestamp(spell["startDate"])
    spellEnd = pd.Timestamp(spell["endDate"])
    for event in events:
        if overlaps(spellStart, spellEnd,
                    pd.Timestamp(event["start"]), pd.Timestamp(event["end"])):
            return event
    return None


if not coldSpells:
    print("nothing detected to corroborate.")
else:
    for spell in coldSpells:
        event = matchToNews(spell, documentedEvents)
        print(f"detected {spell['startDate']} to {spell['endDate']}, "
              f"coldest {spell['coldestF']} F")
        if event:
            print(f"  matches the news: {event['name']} "
                  f"({event['start']} to {event['end']})")
            print(f"  the news reported a low of {event['reportedLowF']} F. {event['note']}")
            print(f"  our data bottomed out at {spell['coldestF']} F.")
            print(f"  the dates line up, and the low is in the same bitter range.")
        else:
            print("  no documented event on these dates. Worth a closer look, or a "
                  "different node or window.")
        print()

## 8. State the finding, fact separate from interpretation

Write your conclusion in two layers, the same discipline the analysis notebook
used.

- **Fact.** In the window studied, the daily minimum stayed at or below the
  severe-cold threshold for the run of days the detector reported, reaching the
  coldest value shown. Those dates overlap a cold event in the independent news
  record.
- **Interpretation.** Attributing that cold to "the polar vortex" is a
  meteorological claim, and it belongs to the National Weather Service and the
  reporting, not to the thermometer. Temperature data can show that it was
  extremely cold for several days on specific dates; it cannot by itself show that
  a stratospheric lobe dislodged. State the cold as your measurement and cite the
  vortex as the explanation others documented.
- **Sensor caveat.** The node's `env.temperature` runs warm, so a measured minimum
  near or above freezing does not mean the outdoor air stayed there. The cold you
  can trust is what a properly sited air sensor reads, or the modeled ambient data
  used offline here. Always know what your thermometer is actually touching.

## 9. Exercises

1. Change the threshold in `detectColdSpells` from -18 C to -12 C (about 10 F).
   Does the detected spell grow, and do any ordinary cold days now get swept in?
   What does that tell you about choosing a threshold?
2. Set `minDays` to 4. Would a genuine but short two-day cold snap now be missed?
   When is a longer minimum the right call, and when is it too strict?
3. Investigate a different event. Point `getWinterData` at late January 2019
   (`2019-01-25` to `2019-02-02`) on a Chicago node and see whether real data is
   available that far back. If it is, does the detected spell match the January 30
   to 31 dates from the news?
4. Add the daily wind chill story: the news lows were driven partly by wind. If a
   node reports a wind metric, pull it for the same window and describe how wind
   would sharpen the felt cold, keeping the measured air temperature separate from
   the felt temperature.
5. Turn the detector into a scanner: pull a whole winter for one node, run
   `detectColdSpells` across it, and list every Arctic outbreak of the season with
   its dates. Which was the most severe?

## Further reading

**The events**

- NWS Chicago, January 30 to 31, 2019 record cold: https://www.weather.gov/lot/RecordColdJan2019
- NWS Chicago, January 19 to 24, 2025 cold spell: https://www.weather.gov/lot/2025_01_19-24_ColdSpell
- 2019 North American cold wave overview: https://en.wikipedia.org/wiki/January%E2%80%93February_2019_North_American_cold_wave

**The science**

- NOAA Climate.gov on the polar vortex: https://www.climate.gov/news-features/understanding-climate/understanding-arctic-polar-vortex

**The tools**

- pandas `resample` for daily aggregation: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html
- matplotlib `axvspan` for shading time ranges: https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.axvspan.html